# 05 Hypothesis and A/B Test

## Business Question

哪些原因可能解释Source 4首次活动后的持续使用差异？如何通过未来随机实验验证可干预的产品假设？

## Analysis Objective

提出H1/H2/H3，区分可进一步数据验证与必须实验验证的问题，选择H2作为优先实验方向，并完成上线前A/B Test设计。本Notebook不生成实验结果。

## Data Used

- `outputs/quality_metrics_post_anomaly_cohort.csv`：Source 4 D7规划基线。
- `outputs/growth_october_daily_validation.csv`：Source 4近期新增流量。
- 当前没有真实assignment、exposure或post-treatment结果。

## Key Metrics

- Primary：D7 Retention。
- Secondary：首7天活跃天数、播放次数、播放时长。
- Long-term：D30 Retention，仅在未来实验cohort成熟后验证。
- Planning Inputs：baseline、MDE、alpha、power、样本量和招募周期。


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
AS_OF_DATE = pd.Timestamp("2015-10-16")
ANALYSIS_START = pd.Timestamp("2015-07-01")
pd.set_option("display.max_columns", 50)


## Hypothesis Framework

### H1：用户来源质量问题

Source 4用户可能具有不同的注册意图或用户构成。需要来源映射、设备、入口、注册前行为和流量真实性数据进一步验证；首次体验实验不能单独确认H1。

### H2：首次体验承接不足

Source 4多数用户能够活动，但后续回访和首周使用较低，可能与首次体验后的内容承接有关。该解释与行为阶段一致，但尚未被证明，需要随机实验验证。

### H3：注册流程或入口差异

10月7日可能存在注册入口、编码、埋点、产品版本或用户迁移变化。需要业务映射和变更日志核查；在确认Source 4代表真实新用户群之前不应上线实验。

## Selected Experimental Hypothesis

选择H2作为优先实验方向，因为它具有产品可干预性，并可使用D7在较短周期内验证。选择H2不等于排除H1或H3。


## Experiment Design

### Population

实验上线后新注册、`registered_via=4`且通过业务映射确认的用户。历史用户不能事后伪随机分组。

### Experimental Unit and Randomization

- 实验单位：`msno`。
- 50/50用户级稳定Hash分流。
- 固定`experiment_id`和salt。
- 同一用户始终处于同一组。
- 主要分析采用Intent-to-Treat。
- 同时记录assignment和真实exposure。

### Control

当前新用户体验流程。

### Treatment

优化首次体验流程，例如简短偏好选择、个性化初始内容和首次播放后的连续内容承接。若多个组件同时上线，实验只能评价整体Treatment效果。

### Metrics

- Primary：D7 Retention。
- Secondary：First 7 Active Days、First 7 Play Count、First 7 Total Seconds。
- Diagnostic：Activation、D1、D3和Treatment曝光率。
- Guardrails：播放失败、异常退出、加载失败、通知退订和应用崩溃。
- Long-term：D30 Retention，仅作为后续长期验证。


In [2]:
quality = pd.read_csv(OUTPUTS_DIR / "quality_metrics_post_anomaly_cohort.csv")
flow = pd.read_csv(OUTPUTS_DIR / "growth_october_daily_validation.csv", parse_dates=["registration_date"])
baseline_d7 = float(quality.loc[quality.registered_via.eq(4), "d7_retention"].iloc[0])
recent_flow = flow[flow.registration_date.between("2015-10-07", "2015-10-16")]
avg_daily_source4 = recent_flow.source4_new_users.mean()
print(f"Source 4 D7 planning baseline: {baseline_d7:.4%}")
print(f"Source 4 average daily new users: {avg_daily_source4:,.1f}")


Source 4 D7 planning baseline: 4.1818%
Source 4 average daily new users: 7,223.5


In [3]:
# Planning assumptions only; MDE is not an observed or promised uplift.
alpha = 0.05
power = 0.80
absolute_mde = 0.003
z_alpha = 1.959963984540054
z_power = 0.8416212335729143
target_rate = baseline_d7 + absolute_mde
p_bar = (baseline_d7 + target_rate) / 2
n_per_group = math.ceil((
    z_alpha * math.sqrt(2*p_bar*(1-p_bar)) +
    z_power * math.sqrt(baseline_d7*(1-baseline_d7) + target_rate*(1-target_rate))
)**2 / absolute_mde**2)
total_sample = 2 * n_per_group
recruitment_days = total_sample / avg_daily_source4
planning = pd.DataFrame([{
    "baseline_d7": baseline_d7, "absolute_mde": absolute_mde,
    "alpha": alpha, "power": power, "allocation": "50/50 two-sided",
    "sample_per_group": n_per_group, "total_sample": total_sample,
    "avg_daily_source4": avg_daily_source4,
    "estimated_recruitment_days": recruitment_days,
    "estimated_complete_d7_days": recruitment_days + 7,
    "estimated_complete_d30_days": recruitment_days + 30,
}])
display(planning)


,baseline_d7,absolute_mde,alpha,power,allocation,sample_per_group,total_sample,avg_daily_source4,estimated_recruitment_days,estimated_complete_d7_days,estimated_complete_d30_days
0,0.041818,0.003,0.05,0.8,50/50 two-sided,72282,144564,7223.5,20.013013,27.013013,50.013013


## Readout and Governance

- 第一批D7在入组后7天成熟；完整D7在最后一批入组用户达到Day7后读取。
- 第一批D30在入组后30天成熟；完整D30只作为长期复核。
- 正式读取前检查样本量、SRM、日志完整性、assignment、exposure及护栏指标。
- 达到预设样本量和成熟窗口后，才计算Control/Treatment差异、置信区间和p值。

## Experiment-design Conclusion

本Notebook完成的是上线前设计。H2仍是待验证假设；当前没有真实实验分组、Treatment曝光、uplift、p值、置信区间或实验成功结论。
